---

## Background

Before diving into the advanced retrieval techniques, let's understand the foundational concepts that make these retrievers powerful.

### What are Advanced Retrievers?

Advanced retrievers in LlamaIndex are sophisticated components that go beyond simple vector similarity search to provide more nuanced, context-aware, and intelligent information retrieval. They combine multiple techniques such as:

- **Semantic Understanding**: Using embeddings to understand meaning and context
- **Keyword Matching**: Precise term-based search for exact specifications
- **Hierarchical Context**: Maintaining relationships between different levels of information
- **Multi-Query Processing**: Generating and combining results from multiple query variations
- **Fusion Techniques**: Intelligently combining results from different retrieval methods

### Why are Advanced Retrievers Important?

1. **Improved Accuracy**: Advanced retrievers can find more relevant information by using multiple search strategies
2. **Better Context Preservation**: They maintain important relationships between pieces of information
3. **Reduced Hallucination**: More precise retrieval leads to more accurate AI responses
4. **Scalability**: Efficient retrieval strategies work better with large document collections
5. **Flexibility**: Different retrieval methods can be combined for optimal results

### Index Types Overview

Before exploring advanced retrievers, it's helpful to first understand the three main index types supported by LlamaIndex. Each is designed to support different retrieval scenarios:

**VectorStoreIndex:**
- Stores vector embeddings for each document chunk
- Best suited for semantic retrieval based on meaning
- Commonly used in LLM pipelines and RAG applications

**DocumentSummaryIndex:**
- Generates and stores summaries of documents at indexing time
- Uses summaries to filter documents before retrieving full content
- Especially useful for large and diverse document sets that cannot fit in the context window of an LLM

**KeywordTableIndex:**
- Extracts keywords from documents and maps them to specific content chunks
- Enables exact keyword matching for rule-based or hybrid search scenarios
- Ideal for applications requiring precise term matching


In [1]:
import os
import json
from typing import List, Optional
import asyncio
import warnings
import numpy as np
warnings.filterwarnings('ignore')

In [2]:
# Core LlamaIndex imports
from llama_index.core import (
    VectorStoreIndex, 
    SimpleDirectoryReader, 
    Document,
    Settings,
    DocumentSummaryIndex,
    KeywordTableIndex
)

In [3]:
from llama_index.core.retrievers import (
    BaseRetriever,
    VectorIndexRetriever,
    AutoMergingRetriever,
    RecursiveRetriever,
    QueryFusionRetriever
)

In [4]:
from llama_index.core.indices.document_summary import (
    DocumentSummaryIndexLLMRetriever,
    DocumentSummaryIndexEmbeddingRetriever,
)

In [5]:
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.core.schema import NodeWithScore, QueryBundle
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.embeddings import BaseEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


In [6]:
# Advanced retriever imports
from llama_index.retrievers.bm25 import BM25Retriever

In [7]:
# Sentence transformers
from sentence_transformers import SentenceTransformer

In [8]:
from langchain_ollama import ChatOllama,OllamaEmbeddings

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [9]:
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import Settings

Settings.embed_model = OllamaEmbedding(
    model_name="mxbai-embed-large:latest",
    base_url="http://localhost:11434",
)

In [10]:
# Sample data for the lab - AI/ML focused documents
SAMPLE_DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.",
    "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "Computer vision allows machines to interpret and understand visual information from the world.",
    "Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.",
    "Supervised learning uses labeled training data to learn a mapping from inputs to outputs.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.",
    "Generative AI can create new content including text, images, code, and more.",
    "Large language models are trained on vast amounts of text data to understand and generate human-like text."
]

# Consistent query examples used throughout the lab
DEMO_QUERIES = {
    "basic": "What is machine learning?",
    "technical": "neural networks deep learning", 
    "learning_types": "different types of learning",
    "advanced": "How do neural networks work in deep learning?",
    "applications": "What are the applications of AI?",
    "comprehensive": "What are the main approaches to machine learning?",
    "specific": "supervised learning techniques"
}

In [11]:
documents=[Document(text=text) for text in SAMPLE_DOCUMENTS]

In [12]:
from pprint import pprint
pprint(documents[0].text)

('Machine learning is a subset of artificial intelligence that focuses on '
 'algorithms that can learn from data.')


In [13]:
nodes=SentenceSplitter().get_nodes_from_documents(documents)

In [14]:
for i in nodes:
    print(i.text)

Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
Natural language processing enables computers to understand, interpret, and generate human language.
Computer vision allows machines to interpret and understand visual information from the world.
Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.
Supervised learning uses labeled training data to learn a mapping from inputs to outputs.
Unsupervised learning finds hidden patterns in data without labeled examples.
Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.
Generative AI can create new content including text, images, code, and more.
Large language models are trained on vast amounts of text data to understand and generate human-like text.


In [15]:
# creating various indexes
vector_index=VectorStoreIndex.from_documents(documents)

In [16]:
from llama_index.llms.ollama import Ollama
Settings.llm = Ollama(
    model="minimax-m3:cloud",
    request_timeout=300,
)

In [17]:
document_summary_index=DocumentSummaryIndex.from_documents(documents)

current doc id: 945179ec-4c49-4385-b779-36f48110a210
current doc id: 89fe36da-b49d-4031-acdc-d477560f62ff
current doc id: 4904610f-8c9d-4cd0-9cab-f66b6a5b9d76
current doc id: 35e8fd7f-28ff-40ce-ae9a-816ce14dff9e
current doc id: a0875c05-ded1-42d4-b770-491eefd9f52d
current doc id: 9438809a-9d65-4ccb-9615-6c805c39e656
current doc id: 771319c9-2085-4ade-bcbd-279f1961f0bb
current doc id: 16b7cc90-82c3-4947-8eb1-a5260285c2e9
current doc id: b78c5106-18a5-46ce-a758-6522c55152b2
current doc id: 6f30aff6-f123-4844-8177-ca861c88d12e


In [18]:
keyword_index=KeywordTableIndex.from_documents(documents)

## 1. Vector Index Retriever - The Foundation

The Vector Index Retriever uses vector embeddings to find semantically related content, making it ideal for general-purpose search and widely used in retrieval-augmented generation (RAG) pipelines.

**How it works**: 
- Documents are split into nodes and embedded using the configured embedding model
- Query is converted to an embedding vector
- Returns nodes ranked by cosine similarity to the query embedding
- Generates embeddings in batches of 2048 nodes by default

**When to use:**
- General-purpose semantic search (most common use case)
- Finding conceptually related content based on meaning rather than exact keywords
- RAG pipelines where semantic understanding is crucial
- When exact keyword matching isn't the primary requirement

**Key characteristics from authoritative source:**
- **Stores embeddings for each document chunk** (VectorStoreIndex foundation)
- **Best for semantic retrieval** based on meaning and context
- **Commonly used in LLM pipelines** for retrieval-augmented generation

**Strengths**: 
- Excellent semantic understanding and context awareness
- Handles synonyms and related concepts effectively
- Works well with natural language queries

**Limitations**: 
- May miss exact keyword matches when specific terms are crucial
- Requires a good embedding model for optimal performance
- Can be computationally intensive for large document collections


In [19]:
# Basic vector retriever
vector_retriever=VectorIndexRetriever(
    index=vector_index,
    similarity_top_k=3
)

In [20]:
query=DEMO_QUERIES["basic"]
query

'What is machine learning?'

In [21]:
node=vector_retriever.retrieve(query)

In [22]:
for i,node in enumerate(node,1):
    print(f"{i}. Score:{node.score:.4f}")
    print(f"  Text: {node.text}")
    print('='*200)

1. Score:0.8630
  Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
2. Score:0.7039
  Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.
3. Score:0.6810
  Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs.


## 2. BM25 Retriever - Advanced Keyword-Based Search

BM25 is a keyword-based retrieval method that improves on TF-IDF by addressing some of its key limitations. It's widely used in production search systems including Elasticsearch and Apache Lucene.

### Understanding TF-IDF: The Foundation

Before diving into BM25, let's understand **TF-IDF** (Term Frequency-Inverse Document Frequency), which BM25 builds upon:

**Term Frequency (TF)**: Measures how often a word appears in a document
- Example: If "neural" appears 3 times in a 100-word document, TF = 3/100 = 0.03

**Inverse Document Frequency (IDF)**: Measures how rare a word is across all documents
- Example: If "neural" appears in only 2 out of 1000 documents, IDF = log(1000/2) = 6.21
- Common words like "the" have low IDF; rare technical terms have high IDF

**TF-IDF Score**: TF × IDF
- Highlights words that are frequent in one document but rare across the collection
- Developed by Karen Spärck Jones, who pioneered the concept of term specificity

### How BM25 Improves Upon TF-IDF

**Key BM25 Improvements:**

1. **Term Frequency Saturation**: BM25 reduces the impact of repeated terms using term frequency saturation
   - Problem: In TF-IDF, if a word appears 100 times vs 10 times, the score increases linearly
   - Solution: BM25 uses a saturation function that plateaus after a certain frequency

2. **Document Length Normalization**: BM25 adjusts for document length, making it more effective for keyword-based search
   - Problem: In TF-IDF, longer documents have unfair advantages
   - Solution: BM25 normalizes scores based on document length relative to average

3. **Tunable Parameters**: Allows fine-tuning for different types of content
   - k1 ≈ 1.2: Controls term frequency saturation (how quickly scores plateau)
   - b ≈ 0.75: Controls document length normalization (0=none, 1=full)

### When to Use BM25

**Ideal for:**
- Technical documentation where exact terms matter
- Legal documents with specific terminology
- Product catalogs with precise specifications
- Academic papers with specialized vocabulary
- Applications requiring keyword-based retrieval rather than semantic similarity

**Advantages:**
- Excellent precision for exact term matches
- Fast computational performance
- Proven effectiveness in production systems
- No training required (unlike neural approaches)
- Interpretable scoring mechanism

**Limitations:**
- No semantic understanding (doesn't handle synonyms)
- Struggles with typos and variations
- Limited context understanding
- Requires careful parameter tuning for optimal performance


In [23]:
import Stemmer

In [24]:
# create BM25 retriever with default parameters
bm25_retriever=BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=3,
    stemmer=Stemmer.Stemmer("english"),
    language="english"
)

In [25]:
query=DEMO_QUERIES['technical']
query

'neural networks deep learning'

In [26]:
node_bm25=bm25_retriever.retrieve(query)

In [27]:
for i,node in enumerate(node_bm25,1):
    print(f"{i}. Score:{node.score:.4f}")
    print(f"  Text: {node.text}")
    
     # Demonstrate TF-IDF concept manually
    text_lower = node.text.lower()
    query_terms = query.lower().split()
    found_terms = [term for term in query_terms if term in text_lower]
        
    if found_terms:
        print(f"   → BM25 would boost this result for terms: {found_terms}")
    print('='*200)
    
print("BM25 Concept Demonstration:")
print("1. TF-IDF Foundation:")
print("   - Term Frequency: How often words appear in document")
print("   - Inverse Document Frequency: How rare words are across collection")
print("   - TF-IDF = TF × IDF (balances frequency vs rarity)")
print()
print("2. BM25 Improvements:")
print("   - Saturation: Prevents over-scoring repeated terms")
print("   - Length normalization: Prevents long document bias")
print("   - Tunable parameters: k1 (saturation) and b (length adjustment)")
print()
print("3. Real-world Usage:")
print("   - Elasticsearch default scoring function")
print("   - Apache Lucene/Solr standard")
print("   - Used in 83% of text-based recommender systems")
print("   - Developed by Robertson & Spärck Jones at City University London")

1. Score:2.5203
  Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
   → BM25 would boost this result for terms: ['neural', 'networks', 'deep', 'learning']
2. Score:0.3372
  Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.
   → BM25 would boost this result for terms: ['learning']
3. Score:0.3024
  Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
   → BM25 would boost this result for terms: ['learning']
BM25 Concept Demonstration:
1. TF-IDF Foundation:
   - Term Frequency: How often words appear in document
   - Inverse Document Frequency: How rare words are across collection
   - TF-IDF = TF × IDF (balances frequency vs rarity)

2. BM25 Improvements:
   - Saturation: Prevents over-scoring repeated terms
   - Length normalization: Prevents long document bias
   - Tunable parameters: k

# BM25 Retriever Workflow

```text
                  Documents
                       │
                       ▼
               Split into Nodes
                       │
                       ▼
               BM25Retriever Index
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
   Term Frequency   Document Length   IDF Statistics
        │              │              │
        └──────────────┼──────────────┘
                       ▼
                BM25 Scoring Engine
                       │
                       ▼
                  User Query
                       │
             Tokenize + Stem Query
                       │
                       ▼
        Compute BM25 Score for Each Node
                       │
                       ▼
            Rank Nodes by BM25 Score
                       │
                       ▼
              Return Top-K Documents
```

---

# Workflow Explanation

## 1. Documents
- Original documents are first divided into smaller chunks (nodes).
- Each node becomes an independent searchable unit.

Example:

```
Document

↓

Node 1
Node 2
Node 3
Node 4
```

---

## 2. Create BM25 Retriever

```python
bm25_retriever = BM25Retriever.from_defaults(
    nodes=lab.nodes,
    similarity_top_k=3,
    stemmer=Stemmer.Stemmer("english"),
    language="english"
)
```

This creates a **BM25 keyword-based retriever**.

Unlike vector search, BM25 does **not** use embeddings.

Instead, it indexes:

- Words
- Word frequencies
- Document lengths
- Inverse document frequencies (IDF)

---

## 3. Stemming

```python
stemmer=Stemmer.Stemmer("english")
```

Before indexing, words are reduced to their root form.

Example:

| Original | Stem |
|-----------|------|
| learning | learn |
| learned | learn |
| learns | learn |
| networks | network |

This improves keyword matching.

---

## 4. User Query

Example

```
"neural networks deep learning"
```

The query is tokenized into:

```
neural

network

deep

learn
```

(after stemming)

---

## 5. BM25 Scoring

Each node receives a BM25 score.

BM25 combines three important factors:

### A. Term Frequency (TF)

How often the query term appears in the document.

Example

```
Node A

neural (4 times)

deep (2 times)
```

Higher frequency generally increases the score.

---

### B. Inverse Document Frequency (IDF)

Rare words are more informative.

Example

```
"the"

appears everywhere

↓

Low IDF
```

```
"backpropagation"

appears in very few documents

↓

High IDF
```

Rare technical terms receive higher importance.

---

### C. Document Length Normalization

Long documents naturally contain more words.

BM25 prevents them from dominating the rankings.

Example

```
Short document
100 words

Long document
3000 words
```

If both contain the same query terms, BM25 adjusts the score so the longer document is not unfairly favored.

---

## 6. Rank Documents

After scoring every node:

```
Node A
Score = 8.73

Node B
Score = 7.94

Node C
Score = 6.81
```

The retriever sorts them in descending order.

---

## 7. Return Top-K Results

```python
similarity_top_k=3
```

Returns only the three highest-scoring nodes.

---

# Why BM25 is Better Than TF-IDF

## TF-IDF Problem 1

Term frequency increases linearly.

```
10 occurrences

↓

Score = 10

100 occurrences

↓

Score = 100
```

Repeated words can dominate the ranking.

---

## BM25 Solution

Uses a **saturation function**.

```
10 occurrences

↓

High score

100 occurrences

↓

Only slightly higher
```

After enough repetitions, additional occurrences contribute very little.

---

## TF-IDF Problem 2

No document length normalization.

Long documents often receive higher scores simply because they contain more words.

---

## BM25 Solution

Applies document length normalization.

```
Short document

↓

Fair score

Long document

↓

Adjusted score
```

This prevents long documents from dominating the results.

---

# Key BM25 Parameters

## k1 (Term Frequency Saturation)

Typical value:

```
k1 ≈ 1.2
```

Controls how quickly term frequency stops increasing the score.

- Small value → Faster saturation
- Large value → Term frequency has more influence

---

## b (Length Normalization)

Typical value:

```
b ≈ 0.75
```

Controls document length normalization.

| Value | Meaning |
|--------|---------|
| 0 | Ignore document length |
| 1 | Full normalization |
| 0.75 | Common default |

---

## IDF Weighting

Rare words receive higher scores than common words.

Example:

```
the

↓

Very low score

transformer

↓

Very high score
```

---

# BM25 Retrieval Process

```text
User Query
      │
      ▼
Tokenize Query
      │
      ▼
Stem Words
      │
      ▼
Compute BM25 Score
      │
      ├─────────── TF
      ├─────────── IDF
      └─────────── Length Normalization
      │
      ▼
Rank Documents
      │
      ▼
Return Top-K Nodes
```

---

# Fallback Workflow (If BM25 Is Not Installed)

If `PyStemmer` is unavailable:

```python
fallback_retriever = lab.vector_index.as_retriever(...)
```

The code switches to a **Vector Retriever**.

Workflow:

```text
Documents
      │
      ▼
Generate Embeddings
      │
      ▼
Vector Store Index
      │
      ▼
User Query
      │
      ▼
Generate Query Embedding
      │
      ▼
Cosine Similarity Search
      │
      ▼
Return Top-K Similar Nodes
```

This fallback is only for demonstration and **does not implement true BM25 scoring**.

---

# BM25 vs Vector Retrieval

| Feature | BM25 | Vector Retrieval |
|----------|-------|------------------|
| Search Type | Keyword matching | Semantic similarity |
| Uses Embeddings | ❌ No | ✅ Yes |
| Handles Synonyms | ❌ Limited | ✅ Excellent |
| Exact Keyword Search | ✅ Excellent | ⚠️ Moderate |
| Semantic Understanding | ❌ No | ✅ Yes |
| Speed | Very Fast | Fast |
| Best Use Case | Search engines, document lookup | RAG systems, question answering |

---

# Overall Workflow

```text
Documents
     │
     ▼
Split into Nodes
     │
     ▼
Build BM25 Index
     │
     ▼
User Query
     │
     ▼
Tokenize + Stem Query
     │
     ▼
Compute BM25 Score
(TF + IDF + Length Normalization)
     │
     ▼
Rank Nodes
     │
     ▼
Return Top-K Relevant Results
```

## 3. Document Summary Index Retrievers

Document Summary Index Retrievers use document summaries instead of the actual documents to find relevant content, making them efficient for large collections. **They return the original documents, not their summaries.**

**How it works (from authoritative source)**:
- **Generates and stores summaries of documents** at indexing time
- **Uses summaries to filter documents** before retrieving full content
- **Two-stage Process**: First uses summaries to filter documents, then returns full document content
- **Especially useful for large, diverse corpora** that cannot fit in the context window of an LLM

**Two Retrieval Options**: 
1. **DocumentSummaryIndexLLMRetriever**: 
   - Uses a large language model to analyze the query against document summaries
   - Provides intelligent document selection but can be more time-consuming and expensive
   - Best for complex queries requiring nuanced understanding

2. **DocumentSummaryIndexEmbeddingRetriever**: 
   - Uses semantic similarity between the query and summary embeddings
   - Faster and more cost-effective than LLM-based approach
   - Good for straightforward similarity matching

**When to use (based on authoritative guidance):**
- Large document collections where documents cover different topics
- When you need efficient document-level filtering before detailed retrieval
- Multi-document QA where documents have distinct subject matters
- Large and diverse document sets that cannot fit in the context window of an LLM

**Configuration Parameters:**
- `choice_top_k` (LLM retriever): Number of documents to select
- `similarity_top_k` (Embedding retriever): Number of documents to select
- Default is 1, increase for multiple document retrieval

**Key Point**: **Returns original documents, not their summaries** - the summaries are only used for filtering

**Strengths**: 
- Efficient document selection and reduces search space
- Good for heterogeneous collections with diverse topics
- Returns original documents with full context intact

**Limitations**: 
- Requires LLM for summary generation during indexing
- May lose some detail present in original documents during summary creation
- LLM-based version can be slower and more expensive than other options


In [28]:
# LLM-based document summary retriever
doc_summary_retriever_llm=DocumentSummaryIndexEmbeddingRetriever(
    document_summary_index,
    choice_tok_k=3 # Number of document to select
)

In [37]:
# Embedding-based document summary retriever
doc_summary_retriever_embedding=DocumentSummaryIndexEmbeddingRetriever(
    document_summary_index,
    similarity_top_k=3 # number of document to select
)

In [29]:
query=DEMO_QUERIES["learning_types"]
query

'different types of learning'

In [31]:
## llm based document summary retriever
nodes_llm=doc_summary_retriever_llm.retrieve(query)
print(f"Retrieved {len(nodes_llm)} nodes")

Retrieved 1 nodes


In [35]:
for i,node in enumerate(nodes_llm[:2],1):
    print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
    print(f"   Text: {node.text}...")
    

1. (Document summary)
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....


In [38]:
# Embedding based document summary retriever
nodes_emb=doc_summary_retriever_embedding.retrieve(query)
print(f"Retrieve {len(nodes_emb)} nodes")

Retrieve 3 nodes


In [39]:
for i, node in enumerate(nodes_emb[:2], 1):
    print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
    print(f"   Text: {node.text[:80]}...")

1. (Document summary)
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to...
2. (Document summary)
   Text: Reinforcement learning is a type of machine learning where agents learn to make ...


In [40]:
print("Document Summary Index workflow:")
print("1. Generates summaries for each document using LLM")
print("2. Uses summaries to select relevant documents")
print("3. Returns full content from selected documents")

Document Summary Index workflow:
1. Generates summaries for each document using LLM
2. Uses summaries to select relevant documents
3. Returns full content from selected documents


# Document Summary Index Retrievers Workflow

```text
                     Documents
                          │
                          ▼
             DocumentSummaryIndex
                          │
          Generate LLM Summary for Each Document
                          │
          ┌───────────────┴───────────────┐
          ▼                               ▼
 Document Summary                 Document Summary
      (Doc 1)                         (Doc 2)
          ▼                               ▼
      Embedding                      Embedding
          │                               │
          └───────────────┬───────────────┘
                          ▼
                Document Summary Index
                          │
          ┌───────────────┴───────────────┐
          ▼                               ▼
LLM Document Summary Retriever    Embedding Summary Retriever
          │                               │
          ▼                               ▼
    Query + LLM Reasoning          Vector Similarity Search
          │                               │
          ▼                               ▼
 Select Most Relevant Docs        Select Most Similar Summaries
          │                               │
          └───────────────┬───────────────┘
                          ▼
            Return Full Document Content
```

---

# Workflow Explanation

## 1. Documents
- Multiple documents are provided as input.
- Each document may contain several pages or sections.

---

## 2. DocumentSummaryIndex
- Creates an index at the **document level** rather than the chunk level.
- Uses an LLM to generate a concise summary for every document.

Example:

```
Document 1
↓

Summary:
"This document explains supervised and unsupervised learning."
```

```
Document 2
↓

Summary:
"This document discusses neural networks and deep learning."
```

---

## 3. Generate Document Summaries
Each document is summarized independently.

```
Document
        │
        ▼
Large Language Model
        │
        ▼
Short Summary
```

These summaries become the searchable representation of the documents.

---

## 4. Build the Document Summary Index
The index stores:

- Document summaries
- Original document references
- Summary embeddings (for embedding retrieval)

Unlike a VectorStoreIndex, this index does **not** retrieve individual chunks first.

---

# Retrieval Methods

The code demonstrates **two different retrieval strategies**.

---

## A. LLM-Based Document Summary Retriever

```python
DocumentSummaryIndexLLMRetriever(...)
```

### Workflow

```text
User Query
      │
      ▼
Large Language Model
      │
Reads all document summaries
      │
Chooses the most relevant summaries
      │
Returns the corresponding full documents
```

### How it works

Instead of comparing embeddings, the LLM reasons over the summaries.

Example:

```
Query:
"What are different learning types?"

↓

Summary 1
"Supervised and Unsupervised Learning"

↓

Summary 2
"Neural Networks"

↓

LLM decides Summary 1 is the best match.
```

### Advantages

- Better semantic reasoning
- Handles complex queries well
- Understands intent beyond keyword similarity

### Disadvantages

- Slower
- Requires LLM inference
- Higher computational cost

---

## B. Embedding-Based Document Summary Retriever

```python
DocumentSummaryIndexEmbeddingRetriever(...)
```

### Workflow

```text
User Query
      │
      ▼
Generate Query Embedding
      │
      ▼
Compare with Summary Embeddings
      │
      ▼
Find Most Similar Summaries
      │
      ▼
Return Corresponding Documents
```

### How it works

Each document summary has its own embedding.

Example:

```
Summary 1
↓

Embedding
↓

[0.21, 0.45, ...]

Summary 2
↓

Embedding
↓

[0.13, 0.82, ...]
```

The query is also converted into an embedding, and cosine similarity is used to find the closest summaries.

### Advantages

- Fast
- Scalable
- No LLM call during retrieval

### Disadvantages

- Relies only on embedding similarity
- May miss nuanced intent

---

# Final Output

Both retrievers ultimately return the **full document content**, not just the summaries.

```text
User Query
      │
      ▼
Document Summary Index
      │
      ├───────────────┐
      ▼               ▼
LLM Retriever   Embedding Retriever
      │               │
      ▼               ▼
Relevant Document Summaries
      │
      ▼
Original Full Documents
      │
      ▼
Returned to the User
```

---

# Comparison

| Feature | LLM-Based Retriever | Embedding-Based Retriever |
|----------|---------------------|---------------------------|
| Selection Method | LLM reasoning | Vector similarity |
| Speed | Slower | Faster |
| Cost | Higher | Lower |
| Semantic Understanding | Excellent | Good |
| Scalability | Moderate | High |
| Best Use Case | Complex or ambiguous queries | Large-scale document retrieval |

---

# Overall Workflow

```text
Documents
     │
     ▼
Generate Document Summaries (LLM)
     │
     ▼
Document Summary Index
     │
     ├───────────────┐
     ▼               ▼
LLM Retriever   Embedding Retriever
     │               │
     ▼               ▼
Select Relevant Documents
     │
     ▼
Return Full Document Content
```

## 4. Auto Merging Retriever - Hierarchical Context Preservation

Auto Merging Retriever is designed to preserve context in long documents using a hierarchical structure. **It uses hierarchical chunking to break documents into parent and child nodes, and if enough child nodes from the same parent are retrieved, the retriever returns the parent node instead.**

**How it works (from authoritative source)**:
- **Uses hierarchical chunking** to break documents into parent and child nodes
- **Retrieves parent if enough children match** - intelligent merging logic
- **Preserves context in long documents** by consolidating related content
- **Dual Storage**: Smaller child chunks are indexed in the vector store for precise matching, while larger parent chunks are stored in the docstore

**Key behavior pattern**:
- Child chunks enable precise matching for specific queries
- When multiple child chunks from the same parent are retrieved, the system returns the parent chunk
- This **helps consolidate related content and preserve broader context**

**When to use (based on authoritative guidance):**
- Long documents where small chunks lose important surrounding context
- Legal documents, research papers, technical specifications that need context preservation
- When you need both precise matching and comprehensive context
- Documents with natural hierarchical structure (sections, subsections)

**Configuration:**
- `chunk_sizes`: List of chunk sizes from largest to smallest (e.g., [512, 256, 128])
- `chunk_overlap`: Overlap between chunks to maintain continuity
- Storage context manages both vector store (child nodes) and docstore (parent nodes)

**Strengths**: 
- Automatically preserves context without manual intervention
- Reduces information fragmentation in long documents
- Intelligent merging based on retrieval patterns
- Maintains granular search capability while providing broader context

**Limitations**: 
- More complex setup compared to basic retrievers
- Requires hierarchical document structure to be effective
- Higher storage overhead due to multiple chunk levels
- May not be suitable for very short documents

*Based on: https://docs.llamaindex.ai/en/stable/examples/retrievers/auto_merging_retriever/*


In [ ]:
## Create hierrarchical nodes 
node_parser=HierarchicalNodeParser.from_defaults(
    chunk_sizes=[512,256,128] # parent,child,chunk
)
hier_nodes=node_parser.get_nodes_from_documents(documents)

In [42]:
# Create storage context with all nodes
from llama_index.core import StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.vector_stores import SimpleVectorStore

In [44]:
docstore=SimpleDocumentStore()
docstore.add_documents(hier_nodes)
storage_context=StorageContext.from_defaults(docstore=docstore)

In [45]:
# Create base index
base_index = VectorStoreIndex(hier_nodes, storage_context=storage_context)
base_retriever = base_index.as_retriever(similarity_top_k=6)


In [46]:
# Create auto-merging retriever
auto_merging_retriever = AutoMergingRetriever(
    base_retriever, 
    storage_context,
    verbose=True
)

In [47]:
query = DEMO_QUERIES["advanced"]  # "How do neural networks work in deep learning?"
nodes = auto_merging_retriever.retrieve(query)


> Merging 1 nodes into parent node.
> Parent node id: 57c1770e-11df-4760-b9c7-2a9ca361026f.
> Parent node text: Deep learning uses neural networks with multiple layers to model and understand complex patterns ...

> Merging 1 nodes into parent node.
> Parent node id: 130a4da5-da31-4a0e-acc1-b17dea6ea472.
> Parent node text: Deep learning uses neural networks with multiple layers to model and understand complex patterns ...

> Merging 1 nodes into parent node.
> Parent node id: 22ad5d8a-4bba-4e84-967a-b39000210428.
> Parent node text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs.

> Merging 1 nodes into parent node.
> Parent node id: b57a1f2a-48bf-4aaa-876d-53b1f6529fdd.
> Parent node text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs.

> Merging 1 nodes into parent node.
> Parent node id: 57c1770e-11df-4760-b9c7-2a9ca361026f.
> Parent node text: Deep learning uses neural networks with multiple layer

In [48]:
print(f"Query: {query}")
print(f"Auto-merged to {len(nodes)} nodes")
for i, node in enumerate(nodes[:3], 1):
    print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Auto-merged)")
    print(f"   Text: {node.text[:120]}...")

Query: How do neural networks work in deep learning?
Auto-merged to 2 nodes
1. Score: 0.8394
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in data....
2. Score: 0.6460
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....


# Auto-Merging Retriever Workflow

```text
                Documents
                     │
                     ▼
      HierarchicalNodeParser
                     │
      ┌──────────────┼──────────────┐
      ▼              ▼              ▼
   512 Nodes      256 Nodes      128 Nodes
      │              │              │
      └──────────────┼──────────────┘
                     ▼
            SimpleDocumentStore
                     │
                     ▼
             StorageContext
                     │
                     ▼
             VectorStoreIndex
                     │
                     ▼
             Base Retriever
                     │
                     ▼
      Top-K Similar Small Chunks
                     │
                     ▼
         AutoMergingRetriever
                     │
     Merge sibling chunks into parents
                     │
                     ▼
      Larger, Context-Rich Retrieved Nodes
```

## Workflow Explanation

1. **Documents**
   - The original documents are provided as input.

2. **HierarchicalNodeParser**
   - Splits each document into multiple hierarchical chunk sizes (e.g., 512, 256, and 128 tokens).
   - Maintains parent-child relationships between chunks.

3. **Hierarchical Nodes**
   - **512-token nodes** (Parent chunks)
   - **256-token nodes** (Child chunks)
   - **128-token nodes** (Leaf chunks)

4. **SimpleDocumentStore**
   - Stores every node along with its metadata and hierarchy.

5. **StorageContext**
   - Holds references to the document store and provides access to the stored nodes.

6. **VectorStoreIndex**
   - Creates embeddings for all hierarchical nodes.
   - Enables semantic similarity search.

7. **Base Retriever**
   - Retrieves the most relevant small chunks using vector similarity search.

8. **Top-K Similar Small Chunks**
   - Returns the top `k` matching chunks (e.g., six 128-token chunks).

9. **AutoMergingRetriever**
   - Examines whether retrieved chunks belong to the same parent.
   - If multiple sibling chunks are retrieved, it merges them into their parent chunk.
   - This process may continue recursively, potentially merging into even larger parent chunks.

10. **Final Output**
    - Returns fewer but larger, context-rich chunks instead of many fragmented snippets.
    - Provides better context for downstream Large Language Models (LLMs).